In [1]:
import os
import glob
import pickle as pkl
from pathlib import Path
import csv
import toml

In [2]:
import os 


combined_data = {}
common_seqs=0

correct_alphabets=["a","c","g","t"]

for pickle_file in os.listdir("./TERRIER_labellingsystems_alldatasets/"):
    if pickle_file.endswith(".pkl"):
        new_pickle_file = os.path.join("./newtraindata/", pickle_file)
        new_dict={}
        print(f"Loading {pickle_file}...")
        data = pkl.load(open(os.path.join("./TERRIER_labellingsystems_alldatasets/", pickle_file), "rb"))
        print(f"Contents of {pickle_file}:")
        print(len(data))
        for seq, info in data.items():
            seq=seq.lower()
            new_seq = [char if char in correct_alphabets else "x" for char in seq]
            seq = "".join(new_seq)
            # if seq not in combined_data:
            label = info["mapped_label"].strip().split("/")
            if len(label) ==2:
                new_dict[seq] = label[0],label[1]
                if seq not in combined_data:
                    combined_data[seq] = label[0],label[1]
                else:
                    common_seqs += 1
            elif len(label) ==1:
                new_dict[seq] = label[0],""
                if seq not in combined_data:
                    combined_data[seq] = label[0],""
                else:
                    common_seqs += 1
            else:
                print(f"Unexpected label format for sequence: {seq[:3]}... Label: {info['mapped_label']}")
        with open(new_pickle_file, "wb") as f:
            pkl.dump(new_dict, f)
print(f"Total unique sequences in combined data: {len(combined_data)}")
print(f"Total common sequences across datasets: {common_seqs}")

Loading RepBase31pt04_converted2_terrierlabelsMay13.pkl...
Contents of RepBase31pt04_converted2_terrierlabelsMay13.pkl:
116787
Loading Repetdb_allfiltered_terrier_May13.pkl...
Contents of Repetdb_allfiltered_terrier_May13.pkl:
73155
Loading MnTEdb_terrier.pkl...
Contents of MnTEdb_terrier.pkl:
5372
Total unique sequences in combined data: 195307
Total common sequences across datasets: 7


## Filter out sequences with greater 20% of "x" composition in them and save new files

In [39]:
train_folder= "./newtraindata/"
new_train_folder= "./newtraindata_filtered/"
os.makedirs(new_train_folder, exist_ok=True)

for pickle_file in os.listdir(train_folder):
    if pickle_file.endswith(".pkl"):

        print(f" {pickle_file}")
        data = pkl.load(open(os.path.join(train_folder, pickle_file), "rb"))
        print(f"Total sequences before filtering: {len(data)}")
        new_data={}
        for seq, labels in data.items():
            # print(f"Sequence: {seq[:30]}... Labels: {labels}")
            percentage_of_x = seq.count("x") / len(seq)
            if percentage_of_x > 0.2:
                # print(f"Skipping sequence due to high percentage of 'x': {percentage_of_x:.2%}")
                pass
            else:
                new_data[seq]=labels
        print(f"Total sequences after filtering: {len(new_data)}")
        new_pickle_file = os.path.join(new_train_folder, pickle_file)
        with open(new_pickle_file, "wb") as f:
            pkl.dump(new_data, f)


 RepBase31pt04_converted2_terrierlabelsMay13.pkl
Total sequences before filtering: 116787
Total sequences after filtering: 116766
 Repetdb_allfiltered_terrier_May13.pkl
Total sequences before filtering: 73155
Total sequences after filtering: 73155
 MnTEdb_terrier.pkl
Total sequences before filtering: 5372
Total sequences after filtering: 4927


### Check outputs

In [41]:
train_folder= "./newtraindata_filtered/"

for pickle_file in os.listdir(train_folder):
    if pickle_file.endswith(".pkl"):
        print(f" {pickle_file}")
        data = pkl.load(open(os.path.join(train_folder, pickle_file), "rb"))
        for seq, labels in list(data.items())[:5]:
            print(f"Sequence: {seq[:30]}... Labels: {labels}")

 RepBase31pt04_converted2_terrierlabelsMay13.pkl
Sequence: tggtgccgtgaccaggatatatagctcgaa... Labels: ('LTR', 'Pao')
Sequence: tgttgtggagacgcagcttcatcatccact... Labels: ('LTR', 'Gypsy')
Sequence: tgtaatcagtatttcgaaactcacaggagt... Labels: ('LTR', 'Gypsy')
Sequence: cccaagtagccttaagaaactctatttgat... Labels: ('DNA', '')
Sequence: tgttgcgaagcgtatatcgcaactcgtttg... Labels: ('LTR', 'Pao')
 Repetdb_allfiltered_terrier_May13.pkl
Sequence: aaatatacttatgtctattttatgaaacat... Labels: ('LTR', 'Copia')
Sequence: atatcgatggttagaaggtaaaagggcgta... Labels: ('LTR', 'Copia')
Sequence: atcttgtattacattttaaaatcttagatt... Labels: ('LTR', 'Copia')
Sequence: acttataagtagggagcggatgtttatgct... Labels: ('LTR', 'Copia')
Sequence: cctcaaaagactactcaaaatttctaacga... Labels: ('LTR', 'Copia')
 MnTEdb_terrier.pkl
Sequence: tgttgatgtcatagctgactgctgacctag... Labels: ('LTR', 'Copia')
Sequence: tgtgaaaaattgagaaagaactagaatatt... Labels: ('LTR', 'Copia')
Sequence: tgtcagaaatataggagtagataggtttag... Labels: ('LTR', 'Copia')
Sequ

# Computing distribution of labels and their counts across datasets

In [ ]:
all_unique_labels={}

for seq, labels in combined_data.items():
    if len(labels) == 2:
        if labels[0] not in all_unique_labels:
            all_unique_labels[labels[0]]=[]
        if labels[1] not in all_unique_labels[labels[0]]:
            all_unique_labels[labels[0]].append(labels[1])

    elif len(labels) == 1:
        if labels[0] not in all_unique_labels:
            all_unique_labels[labels[0]]=[]
            
from pprint import pprint
print("Unique labels and their associated sub-labels:")

for label, sub_labels in all_unique_labels.items():
    sub_labels=[x for x in sub_labels if x!=""]
    print(f"'{label}': {sub_labels},")
    # print(label,":", [x.strip() for x in sub_labels])

Unique labels and their associated sub-labels:
'LTR': ['Pao', 'Gypsy', 'Copia', 'DIRS', 'Caulimovirus', 'ERV'],
'DNA': ['Harbinger', 'CMC', 'P', 'hAT', 'TcMar', 'PiggyBac', 'Zator', 'MULE', 'Merlin', 'Kolobok', 'Maverick', 'Novosib', 'Zisupton', 'Crypton', 'Academ', 'IS3EU', 'Dada', 'Sola', 'Ginger'],
'LINE': ['R1', 'I', 'CR1', 'L1', 'RTE', 'L2', 'Dong-R4', 'R2', 'Dualen', 'CRE', 'Tad1', 'Rex-Babar', 'Proto2', 'Proto1'],
'Satellite': [],
'RC': ['Helitron'],
'SINE': ['tRNA', '5S', '7SL', 'U'],
'Structural_RNA': [],
'PLE': [],
'Other': [],


LTR: [Pao, Gypsy, Copia, DIRS, Caulimovirus, ERV]
DNA: [Harbinger, CMC, P, hAT, TcMar, PiggyBac, Zator, MULE, Merlin, Kolobok, Maverick, Novosib, Zisupton, Crypton, Academ, IS3EU, Dada, Sola, Ginger]
LINE: [R1, I, CR1, L1, RTE, L2, Dong-R4, R2, Dualen, CRE, Tad1, Rex-Babar, Proto2, Proto1]
Satellite: []
RC: [Helitron]
SINE: [tRNA, 5S, 7SL, U]
Structural_RNA: []
PLE: []
Other: []

tree={'LTR': ['Pao', 'Gypsy', 'Copia', 'DIRS', 'Caulimovirus', 'ERV'],
'DNA': ['Harbinger', 'CMC', 'P', 'hAT', 'TcMar', 'PiggyBac', 'Zator', 'MULE', 'Merlin', 'Kolobok', 'Maverick', 'Novosib', 'Zisupton', 'Crypton', 'Academ', 'IS3EU', 'Dada', 'Sola', 'Ginger'],
'LINE': ['R1', 'I', 'CR1', 'L1', 'RTE', 'L2', 'Dong-R4', 'R2', 'Dualen', 'CRE', 'Tad1', 'Rex-Babar', 'Proto2', 'Proto1'],
'Satellite': [],
'RC': ['Helitron'],
'SINE': ['tRNA', '5S', '7SL', 'U'],
'Structural_RNA': [],
'PLE': [],
'Other': [],
}

In [ ]:
all_unique_labels_andcounts={}

order_level_counts={}
for seq, labels in combined_data.items():
    if len(labels) == 2:
        if labels[0] not in all_unique_labels_andcounts:
            order_level_counts[labels[0]]=0
            all_unique_labels_andcounts[labels[0]]={}
        order_level_counts[labels[0]] += 1
        if labels[1] not in all_unique_labels_andcounts[labels[0]]:
            all_unique_labels_andcounts[labels[0]][labels[1]] = 0
        all_unique_labels_andcounts[labels[0]][labels[1]] += 1
    elif len(labels) == 1:
        if labels[0] not in all_unique_labels_andcounts:
            order_level_counts[labels[0]]=0
            all_unique_labels_andcounts[labels[0]]={}
        order_level_counts[labels[0]] += 1
    else:
        print(f"Unexpected label format for sequence: {seq[:3]}... Label: {labels}")

for label, sub_labels in all_unique_labels_andcounts.items():
    print(f"Total count {order_level_counts[label]}  '{label}': {sub_labels}")


Total count 145644  'LTR': {'Pao': 8567, 'Gypsy': 84435, 'Copia': 38905, '': 2050, 'DIRS': 2029, 'Caulimovirus': 216, 'ERV': 9442}
Total count 29285  'DNA': {'': 2371, 'Harbinger': 3368, 'CMC': 1836, 'P': 348, 'hAT': 9145, 'TcMar': 5409, 'PiggyBac': 701, 'Zator': 113, 'MULE': 2759, 'Merlin': 187, 'Kolobok': 972, 'Maverick': 264, 'Novosib': 9, 'Zisupton': 46, 'Crypton': 296, 'Academ': 657, 'IS3EU': 128, 'Dada': 170, 'Sola': 413, 'Ginger': 93}
Total count 13171  'LINE': {'R1': 419, 'I': 1169, 'CR1': 1334, 'L1': 5448, 'RTE': 1765, 'L2': 1088, 'Dong-R4': 67, '': 444, 'R2': 345, 'Dualen': 13, 'CRE': 110, 'Tad1': 689, 'Rex-Babar': 180, 'Proto2': 90, 'Proto1': 10}
Total count 574  'Satellite': {'': 574}
Total count 2176  'RC': {'Helitron': 2176}
Total count 2891  'SINE': {'': 90, 'tRNA': 2547, '5S': 43, '7SL': 192, 'U': 19}
Total count 34  'Structural_RNA': {'': 34}
Total count 1521  'PLE': {'': 1521}
Total count 11  'Other': {'': 11}


In [ ]:
train_folder= "./newtraindata_filtered/"

for pickle_file in os.listdir(train_folder):
    if pickle_file.endswith(".pkl"):
        print(f" {pickle_file}")
        data = pkl.load(open(os.path.join(train_folder, pickle_file), "rb"))

        all_unique_labels_andcounts={}

        order_level_counts={}
        for seq, labels in data.items():
            if len(labels) == 2:
                if labels[0] not in all_unique_labels_andcounts:
                    order_level_counts[labels[0]]=0
                    all_unique_labels_andcounts[labels[0]]={}
                order_level_counts[labels[0]] += 1
                if labels[1] not in all_unique_labels_andcounts[labels[0]]:
                    all_unique_labels_andcounts[labels[0]][labels[1]] = 0
                all_unique_labels_andcounts[labels[0]][labels[1]] += 1
            elif len(labels) == 1:
                if labels[0] not in all_unique_labels_andcounts:
                    order_level_counts[labels[0]]=0
                    all_unique_labels_andcounts[labels[0]]={}
                order_level_counts[labels[0]] += 1
            else:
                print(f"Unexpected label format for sequence: {seq[:3]}... Label: {labels}")

        for label, sub_labels in all_unique_labels_andcounts.items():

            print(f"Total count {order_level_counts[label]}  '{label}': {sub_labels}")

        print("==="*20,"\n")
        print("==="*20)



 RepBase31pt04_converted2_terrierlabelsMay13.pkl
Total count 69542  'LTR': {'Pao': 8566, 'Gypsy': 36310, 'Copia': 11906, '': 1074, 'DIRS': 2028, 'Caulimovirus': 216, 'ERV': 9442}
Total count 26946  'DNA': {'': 2371, 'Harbinger': 2812, 'CMC': 1740, 'P': 348, 'hAT': 7672, 'TcMar': 5204, 'PiggyBac': 701, 'Zator': 113, 'MULE': 2758, 'Merlin': 187, 'Kolobok': 972, 'Maverick': 264, 'Novosib': 9, 'Zisupton': 46, 'Crypton': 296, 'Academ': 652, 'IS3EU': 126, 'Dada': 169, 'Sola': 413, 'Ginger': 93}
Total count 13110  'LINE': {'R1': 419, 'I': 1131, 'CR1': 1334, 'L1': 5429, 'RTE': 1762, 'L2': 1088, 'Dong-R4': 67, '': 444, 'R2': 345, 'Dualen': 13, 'CRE': 109, 'Tad1': 689, 'Rex-Babar': 180, 'Proto2': 90, 'Proto1': 10}
Total count 574  'Satellite': {'': 574}
Total count 2137  'RC': {'Helitron': 2137}
Total count 2891  'SINE': {'': 90, 'tRNA': 2547, '5S': 43, '7SL': 192, 'U': 19}
Total count 34  'Structural_RNA': {'': 34}
Total count 1521  'PLE': {'': 1521}
Total count 11  'Other': {'': 11}

 Repetdb_

### Saved counts Over Orders and their superfamilies
<!-- RepBase31pt04_converted2_terrierlabelsMay13.pkl
Total count 69542  'LTR': {'Pao': 8566, 'Gypsy': 36310, 'Copia': 11906, '': 1074, 'DIRS': 2028, 'Caulimovirus': 216, 'ERV': 9442}
Total count 26946  'DNA': {'': 2371, 'Harbinger': 2812, 'CMC': 1740, 'P': 348, 'hAT': 7672, 'TcMar': 5204, 'PiggyBac': 701, 'Zator': 113, 'MULE': 2758, 'Merlin': 187, 'Kolobok': 972, 'Maverick': 264, 'Novosib': 9, 'Zisupton': 46, 'Crypton': 296, 'Academ': 652, 'IS3EU': 126, 'Dada': 169, 'Sola': 413, 'Ginger': 93}
Total count 13110  'LINE': {'R1': 419, 'I': 1131, 'CR1': 1334, 'L1': 5429, 'RTE': 1762, 'L2': 1088, 'Dong-R4': 67, '': 444, 'R2': 345, 'Dualen': 13, 'CRE': 109, 'Tad1': 689, 'Rex-Babar': 180, 'Proto2': 90, 'Proto1': 10}
Total count 574  'Satellite': {'': 574}
Total count 2137  'RC': {'Helitron': 2137}
Total count 2891  'SINE': {'': 90, 'tRNA': 2547, '5S': 43, '7SL': 192, 'U': 19}
Total count 34  'Structural_RNA': {'': 34}
Total count 1521  'PLE': {'': 1521}
Total count 11  'Other': {'': 11}
============================================================ 

============================================================
 Repetdb_allfiltered_terrier_May13.pkl
Total count 72154  'LTR': {'Copia': 25444, 'Gypsy': 46710}
Total count 961  'DNA': {'CMC': 98, 'TcMar': 206, 'hAT': 386, 'Harbinger': 271}
Total count 38  'LINE': {'I': 38}
Total count 2  'RC': {'Helitron': 2}
============================================================ 

============================================================
 MnTEdb_terrier.pkl
Total count 3564  'LTR': {'Copia': 1426, 'Gypsy': 1298, '': 840}
Total count 1313  'DNA': {'hAT': 1041, 'Harbinger': 272}
Total count 31  'RC': {'Helitron': 31}
Total count 19  'LINE': {'L1': 19}
============================================================ 

============================================================ -->